# LFM2-2.6B — J-lens fit (FP16 only)

**Scope of this notebook:** fit `lenses/lfm2-2.6b.pt` on Colab in **float16**.  
**Not in this notebook:** baseline regeneration, Exp1–3 (those run on your laptop after you download the lens).

| Setting | Value |
|---|---|
| Model | `LiquidAI/LFM2-2.6B` (public; not Antidoom; not private LFM2.5) |
| Fit dtype | **fp16** |
| Layers | **full_attention only** (skip LIV conv) |
| Corpus | WikiText-103, default **100** prompts (usable per paper; raise to 300 if time left) |
| seq / skip | `max_seq_len=128`, `skip_first=16` |

**Runtime:** Colab GPU (T4/L4/A100). Prefer **Runtime → Change runtime type → GPU**.  
**Time budget (~4h):** start with 100 prompts; checkpoint every prompt so you can resume after disconnect.

After success: download `lfm2-2.6b.pt` → put in laptop `j-lens/lenses/` → local NF4 Exp1–3.

In [ ]:
# Cell 1 — GPU check + mount Drive (lens + checkpoints survive session death)
import torch
from google.colab import drive
from pathlib import Path

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {props.total_memory/1e9:.1f} GB")
print("CUDA:", torch.version.cuda)

drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/jlens_lfm2_fit_fp16")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("Drive out:", DRIVE_OUT)

In [ ]:
# Cell 2 — deps + anthropic jlens (path inject; no slow pip -e)
import os, sys, shutil, subprocess
from pathlib import Path

%pip install -q -U "transformers>=4.51" accelerate datasets huggingface_hub tqdm safetensors sentencepiece

JLENS_ROOT = Path("/content/jacobian-lens")
MARKER = JLENS_ROOT / "jlens" / "__init__.py"
if not MARKER.is_file():
    if JLENS_ROOT.exists():
        shutil.rmtree(JLENS_ROOT, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/anthropics/jacobian-lens.git", str(JLENS_ROOT)],
        check=True,
    )

if str(JLENS_ROOT) not in sys.path:
    sys.path.insert(0, str(JLENS_ROOT))

import jlens
from jlens.hf import Layout, from_hf
from jlens.examples import load_wikitext_prompts
print("jlens OK", getattr(jlens, "__file__", JLENS_ROOT))

In [ ]:
# Cell 3 — fit knobs (T4 15GB: dim_batch=32 OOMs; use 4)
import os
from pathlib import Path

MODEL_ID = "LiquidAI/LFM2-2.6B"
N_PROMPTS = 100          # paper: ~100 usable; 300 if >2h left after smoke
DIM_BATCH = 4            # jlens expands batch=DIM_BATCH with retain_graph; 32 OOMs on T4
MAX_SEQ_LEN = 128        # drop to 64 only if DIM_BATCH=4 still OOMs
DTYPE_NAME = "fp16"      # FP16 only (this notebook)

WORK = Path("/content/lfm2_jlens_fit")
WORK.mkdir(parents=True, exist_ok=True)
LENS_LOCAL = WORK / "lfm2-2.6b.pt"
CKPT_LOCAL = WORK / "jlens_fit_ckpt.pt"
STATUS_LOCAL = WORK / "jlens_fit_status.json"

# Persistent copies on Drive
LENS_DRIVE = DRIVE_OUT / "lfm2-2.6b.pt"
CKPT_DRIVE = DRIVE_OUT / "jlens_fit_ckpt.pt"
STATUS_DRIVE = DRIVE_OUT / "jlens_fit_status.json"

# Resume from Drive if a previous session got partway
if CKPT_DRIVE.is_file() and not CKPT_LOCAL.is_file():
    import shutil
    shutil.copy2(CKPT_DRIVE, CKPT_LOCAL)
    print("Restored checkpoint from Drive")
if LENS_DRIVE.is_file() and not LENS_LOCAL.is_file():
    import shutil
    shutil.copy2(LENS_DRIVE, LENS_LOCAL)
    print("Restored finished lens from Drive — skip fit if you only need to sanity-check")

print({
    "MODEL_ID": MODEL_ID,
    "N_PROMPTS": N_PROMPTS,
    "DIM_BATCH": DIM_BATCH,
    "MAX_SEQ_LEN": MAX_SEQ_LEN,
    "DTYPE": DTYPE_NAME,
    "ckpt_exists": CKPT_LOCAL.is_file(),
    "lens_exists": LENS_LOCAL.is_file(),
})

In [ ]:
# Cell 4 — SMOKE TEST (1 prompt). Stop here if this OOMs — lower DIM_BATCH and retry.
import gc
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

dtype = torch.float16
assert DTYPE_NAME == "fp16"
torch.cuda.empty_cache()
gc.collect()

hf_config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
layer_types = list(getattr(hf_config, "layer_types", []) or [])
source_layers = [i for i, t in enumerate(layer_types) if "attention" in str(t).lower()]
if not source_layers:
    source_layers = [2, 5, 9, 13, 17, 21, 24, 27]
print("attn source_layers:", source_layers)
print("DIM_BATCH", DIM_BATCH, "MAX_SEQ_LEN", MAX_SEQ_LEN)

# Reuse model if Cell 4 already loaded it earlier in this session
if "model" not in globals() or model is None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=dtype,  # transformers >=4.56; replaces torch_dtype
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    layout = Layout("model", layers="layers", norm="embedding_norm", embed="embed_tokens")
    lm = from_hf(model, tokenizer, layout=layout, force_bos=False)

print("model loaded", type(model).__name__, "params~2.6B fp16")
print("VRAM allocated GB", round(torch.cuda.memory_allocated()/1e9, 2))
print(lm)

smoke_prompts = load_wikitext_prompts(1)
print("smoke prompt chars", len(smoke_prompts[0]))

smoke_ckpt = WORK / "smoke_ckpt.pt"
if smoke_ckpt.is_file():
    smoke_ckpt.unlink()

try:
    smoke_lens = jlens.fit(
        lm,
        smoke_prompts,
        source_layers=source_layers,
        dim_batch=DIM_BATCH,
        max_seq_len=MAX_SEQ_LEN,
        checkpoint_path=str(smoke_ckpt),
        checkpoint_every=1,
        resume=False,
    )
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    gc.collect()
    raise RuntimeError(
        f"OOM at DIM_BATCH={DIM_BATCH}. "
        "Runtime→Restart session, set DIM_BATCH=2 (or 1) in Cell 3, re-run from Cell 1."
    ) from None

print("SMOKE OK — layers fitted:", smoke_lens.source_layers)
print("VRAM peak GB", round(torch.cuda.max_memory_allocated()/1e9, 2))
del smoke_lens
torch.cuda.empty_cache()
gc.collect()
print("Proceed to Cell 5 for the full fit.")

In [ ]:
# Cell 5 — FULL FIT (resumable). Copies checkpoint → Drive every prompt via jlens ckpt + periodic sync.
import json, shutil, time
import torch

if LENS_LOCAL.is_file() and LENS_LOCAL.stat().st_size > 1_000_000:
    print("Lens already exists:", LENS_LOCAL, LENS_LOCAL.stat().st_size)
    print("Skip fit — go to Cell 6 sanity check. Delete the file to refit.")
else:
    prompts = load_wikitext_prompts(N_PROMPTS)
    print(f"Fitting on {len(prompts)} prompts | dim_batch={DIM_BATCH} | layers={source_layers}")
    t0 = time.time()

    lens = jlens.fit(
        lm,
        prompts,
        source_layers=source_layers,
        dim_batch=DIM_BATCH,
        max_seq_len=MAX_SEQ_LEN,
        checkpoint_path=str(CKPT_LOCAL),
        checkpoint_every=1,
        resume=True,
    )
    lens.save(str(LENS_LOCAL))
    elapsed = time.time() - t0
    print(f"Saved {LENS_LOCAL} ({LENS_LOCAL.stat().st_size/1e6:.1f} MB) in {elapsed/60:.1f} min")

    status = {
        "model_id": MODEL_ID,
        "hybrid": True,
        "fitted": True,
        "dtype": "fp16",
        "n_prompts": len(prompts),
        "dim_batch": DIM_BATCH,
        "max_seq_len": MAX_SEQ_LEN,
        "source_layers": list(source_layers),
        "lens_path": str(LENS_LOCAL),
        "elapsed_min": round(elapsed / 60, 2),
        "gpu": torch.cuda.get_device_name(0),
        "note": "Attn-only Jacobian fit. Use with NF4 generation locally; rent larger GPU for full fp16 redo if Exp1–3 look promising.",
    }
    STATUS_LOCAL.write_text(json.dumps(status, indent=2), encoding="utf-8")
    print(json.dumps(status, indent=2))

    # Persist to Drive
    shutil.copy2(LENS_LOCAL, LENS_DRIVE)
    if CKPT_LOCAL.is_file():
        shutil.copy2(CKPT_LOCAL, CKPT_DRIVE)
    shutil.copy2(STATUS_LOCAL, STATUS_DRIVE)
    print("Copied to Drive:", DRIVE_OUT)

In [ ]:
# Cell 5b — OPTIONAL mid-fit Drive sync (run in another cell / periodically if worried about disconnect)
# Safe to re-run anytime while Cell 5 is NOT running, or after interrupt.
import shutil
from pathlib import Path

if CKPT_LOCAL.is_file():
    shutil.copy2(CKPT_LOCAL, CKPT_DRIVE)
    print("Synced ckpt →", CKPT_DRIVE, CKPT_DRIVE.stat().st_size)
else:
    print("No local ckpt yet")
if LENS_LOCAL.is_file():
    shutil.copy2(LENS_LOCAL, LENS_DRIVE)
    print("Synced lens →", LENS_DRIVE)

In [ ]:
# Cell 6 — sanity: final-layer identity-ish readout vs model logits (on an attn layer + last layer if present)
import torch
from jlens.lens import JacobianLens

assert LENS_LOCAL.is_file(), "No lens file — Cell 5 did not finish"
lens = JacobianLens.load(str(LENS_LOCAL))
print("loaded lens", lens)

prompt = "The capital of France is"
lens_logits, model_logits, input_ids = lens.apply(
    lm, prompt, layers=[source_layers[-1]], positions=[-1], max_seq_len=64
)
layer = source_layers[-1]
j_top = lens_logits[layer][0].topk(5).indices.tolist()
m_top = model_logits[0].topk(5).indices.tolist()
print("prompt:", prompt)
print("jlens top5 @ last attn layer", layer, [tokenizer.decode([i]) for i in j_top])
print("model  top5", [tokenizer.decode([i]) for i in m_top])
print("overlap", len(set(j_top) & set(m_top)), "/ 5")
print("\nDONE. Download from Drive:", LENS_DRIVE)
print("Laptop path target: j-lens/lenses/lfm2-2.6b.pt")

## After Colab

1. Confirm `MyDrive/jlens_lfm2_fit_fp16/lfm2-2.6b.pt` exists.
2. Download it to your laptop as `j-lens/lenses/lfm2-2.6b.pt`.
3. Tell the agent in Cursor — local next steps: workspace band → Exp1 → Exp2 analyze → Exp3 (NF4 model + this fp16 lens).
4. You can **close this Colab session** after the Drive copy succeeds.

If Cell 4 smoke OOMs (common on T4 with DIM_BATCH=32): **Runtime → Restart session**, set `DIM_BATCH = 4` (or `2` / `1`) in Cell 3, re-run from Cell 1. Do not keep the fragmented GPU from the failed smoke.
If Cell 5 is slow but progressing: leave it; re-run Cell 1–3 + 5 after disconnect (resume=True). Lower DIM_BATCH only slows fit (more backward passes); it does not change lens meaning.